# Deep Q-Network (DQN) for Cart-Pole Balancing

This notebook demonstrates how to train a Reinforcement Learning agent to balance a pole on a cart using the **Deep Q-Network (DQN)** algorithm implemented in PyTorch, using the Farama Gymnasium environment.

## 1. Import Dependencies
First, we import the necessary libraries. We will use `gymnasium` for the environment, `torch` for building and training our deep neural network, `numpy` for array operations, and `matplotlib` for plotting our results.

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import matplotlib.pyplot as plt
import random
import os

# Import our agent class and helper functions
from dqn_agent import DQNAgent

print(f"PyTorch version: {torch.__version__}")
print(f"Gymnasium version: {gym.__version__}")

## 2. Understand the CartPole-v1 Environment
A pole is attached by an un-actuated joint to a cart, which moves along a frictionless track. The system is controlled by applying a force of +1 or -1 to the cart. The pendulum starts upright, and the goal is to prevent it from falling over.

* **State space** contains 4 continuous observations:
  1. Cart Position (-4.8 to 4.8)
  2. Cart Velocity (-inf to inf)
  3. Pole Angle (-24 deg to 24 deg)
  4. Pole Angular Velocity (-inf to inf)

* **Action space** is discrete with 2 options:
  * `0`: Push cart to the left
  * `1`: Push cart to the right

* **Rewards**: A reward of `+1` is provided for every timestep that the pole remains upright.

* **Episode Termination**: The episode ends if the pole angle is more than 12 degrees from vertical, or if the cart moves more than 2.4 units from the center, or if the step limit of 500 is reached.

In [ ]:
env = gym.make("CartPole-v1")
print(f"Observation Space: {env.observation_space}")
print(f"Action Space: {env.action_space}")

# Reset environment to see initial state
state, info = env.reset()
print(f"Initial State Vector: {state}")
env.close()

## 3. Train the DQN Agent
Now we set up hyperparameters and start training the agent. The target is to reach a 100-episode rolling average reward of at least 475.0 (the standard threshold to solve the CartPole-v1 environment).

In [ ]:
# Set seeds for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Create the training environment
env = gym.make("CartPole-v1")
env.action_space.seed(seed)

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

# Hyperparameters
learning_rate = 1e-3
gamma = 0.99
epsilon_start = 1.0
epsilon_end = 0.01
epsilon_decay = 0.995
buffer_size = 20000
batch_size = 128
tau = 0.005
num_episodes = 300  # We set 300 to solve quickly
max_steps = 500

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

# Initialize DQN Agent
agent = DQNAgent(
    state_dim=state_dim,
    action_dim=action_dim,
    lr=learning_rate,
    gamma=gamma,
    epsilon_start=epsilon_start,
    epsilon_end=epsilon_end,
    epsilon_decay=epsilon_decay,
    buffer_size=buffer_size,
    batch_size=batch_size,
    tau=tau,
    device=device
)

rewards_history = []
avg_rewards_history = []
solved = False

print("Training in progress...")
for episode in range(1, num_episodes + 1):
    state, info = env.reset(seed=seed + episode)
    episode_reward = 0
    
    for step in range(max_steps):
        action = agent.select_action(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        agent.step(state, action, reward, next_state, done)
        state = next_state
        episode_reward += reward
        
        if done:
            break
            
    rewards_history.append(episode_reward)
    running_avg = np.mean(rewards_history[-100:])
    avg_rewards_history.append(running_avg)
    
    if episode % 10 == 0:
        print(f"Episode {episode:<3} | Steps: {step+1:<3} | Reward: {episode_reward:.1f} | Avg (100 ep): {running_avg:.1f} | Epsilon: {agent.epsilon:.3f}")
        
    if running_avg >= 475.0 and episode >= 100:
        print(f"\n[!] Solved in {episode} episodes! Final Running Average Reward: {running_avg:.1f}")
        solved = True
        break

env.close()

# Save trained agent
os.makedirs("models", exist_ok=True)
agent.save("models/cartpole_dqn.pth")
print("\n[+] Saved model weights to models/cartpole_dqn.pth")

## 4. Plot Training Progress
Let's visualize the reward convergence curve over the training episodes.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(rewards_history, label="Episode Reward", color="skyblue", alpha=0.8)
plt.plot(avg_rewards_history, label="100-Episode Moving Avg", color="navy", linewidth=2)
plt.axhline(y=475.0, color="red", linestyle="--", label="Solved Threshold (475)")
plt.title("Cart-Pole DQN Agent Training curve")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Evaluate the Agent
We load the saved model and run evaluation episodes with greedy actions (no exploration) to see how consistently the agent balances the pole.

In [ ]:
# Create env without rendering for quick calculation
eval_env = gym.make("CartPole-v1")
eval_agent = DQNAgent(state_dim=state_dim, action_dim=action_dim, device="cpu")
eval_agent.load("models/cartpole_dqn.pth")

eval_episodes = 5
eval_rewards = []

for ep in range(1, eval_episodes + 1):
    state, info = eval_env.reset(seed=100 + ep)
    episode_reward = 0
    
    for step in range(max_steps):
        action = eval_agent.select_action(state, evaluate=True)
        next_state, reward, terminated, truncated, info = eval_env.step(action)
        done = terminated or truncated
        state = next_state
        episode_reward += reward
        if done:
            break
            
    eval_rewards.append(episode_reward)
    print(f"Eval Episode {ep} | Reward: {episode_reward:.1f}")

eval_env.close()
print(f"\nAverage Eval Reward: {np.mean(eval_rewards):.1f} / {max_steps:.1f}")